# E004 — CPU Predictive Models

**Input checklist**
- Required: `turns.parquet`, `archetype_profiles.parquet`, `macro_library.json`.
- Accelerator: **None / CPU**.
- Internet: **OFF**.
- Models: regularized logistic/ridge plus nearest-centroid archetypes, exported to tiny pure-Python inference.

**Primary target:** opponent SELL volume over the next 24 turns, because pre-empting a premium-resource flood is reversible and lower risk than changing the farm.

In [ ]:
from pathlib import Path
import os,sys,json
SUITE_CANDIDATES=[Path('/kaggle/input/kaggriculture-v2-suite'),Path('/kaggle/input/kaggriculture-v2-suite/kaggriculture_v2_suite'),Path.cwd().parent,Path.cwd()]
ROOT=next((p for p in SUITE_CANDIDATES if (p/'src'/'kagv2').exists()),None)
if ROOT is None: raise FileNotFoundError('Attach/upload kaggriculture_v2_suite as a Kaggle Dataset, or run this notebook inside the repo.')
sys.path.insert(0,str(ROOT)); WORK=Path('/kaggle/working/kagv2') if Path('/kaggle/working').exists() else ROOT/'artifacts'; WORK.mkdir(parents=True,exist_ok=True)
print('ROOT=',ROOT,'WORK=',WORK)

In [ ]:
import pandas as pd,numpy as np,json
from sklearn.preprocessing import StandardScaler
from src.kagv2.models import train_win_model,train_supply_model,save_model_bundle
from src.kagv2.features import public_feature_frame
from src.kagv2.constants import PUBLIC_RUNTIME_FEATURES
turns=pd.read_parquet(WORK/'turns.parquet');profiles=pd.read_parquet(WORK/'archetype_profiles.parquet');lib=json.loads((WORK/'macro_library.json').read_text())
win,wm=train_win_model(turns);supply,sm=train_supply_model(turns,horizon=24,alpha=10)
print('win metrics',wm);print('supply metrics',sm)

In [ ]:
labels=profiles[['episode_id','player','archetype']].drop_duplicates();d=turns.merge(labels,on=['episode_id','player'],how='inner')
d=d[(d.day.between(6,14)) & (d.hour.isin([0,6,12,18]))].copy();X=public_feature_frame(d)
sc=StandardScaler().fit(X);Z=sc.transform(X);centroids=[]
for cl in sorted(d.archetype.unique()):centroids.append(Z[d.archetype.to_numpy()==cl].mean(0).tolist())
arch={'runtime_features':list(X.columns),'mean':sc.mean_.tolist(),'scale':sc.scale_.tolist(),'centroids':centroids}
bundle=save_model_bundle(WORK/'learned_model.json',win=win,supply=supply,archetype=arch,macro_library=lib)
print('model bytes',(WORK/'learned_model.json').stat().st_size)

In [ ]:
assert set(supply['features'])==set(PUBLIC_RUNTIME_FEATURES)
assert not any('shed' in f or 'seed_' in f for f in supply['features'])
print('Top win coefficients:')
coef=pd.Series(win['coef'],index=win['features']).sort_values(key=np.abs,ascending=False);display(coef.head(20))

### Promotion gate
Do not deploy prediction just because training metrics are nonzero. Require stable actor-grouped validation and use predictions selectively. V2 only pre-sells when predicted premium-resource pressure is large and the current price is still healthy.